# BETO — Nested CV 5×3 con grupos · Optuna TPE20 · curvas y checkpoints


## 1. Dependencias e imports

In [ ]:
import os
import gc
import re
import json
import math
import time
import random
import inspect
import pickle
import shutil
import hashlib
import html
import warnings
from pathlib import Path
from datetime import datetime
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import optuna
import emoji
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, Subset
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
)
from sklearn.utils.class_weight import compute_class_weight

import transformers
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    TrainerCallback,
    set_seed,
)
from transformers.trainer_callback import PrinterCallback, ProgressCallback
from transformers.trainer_utils import get_last_checkpoint
from IPython.display import display, HTML

warnings.filterwarnings("ignore", category=FutureWarning)
transformers.logging.set_verbosity_error()
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Optuna:", optuna.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM total (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    print("BF16 soportado:", torch.cuda.is_bf16_supported())

## 2. Configuración principal

In [ ]:
MODEL_NAME = "dccuchile/bert-base-spanish-wwm-cased"

# Archivos creados por el preprocessing Transformer.
DATA_DIR = Path("../data/transformers")
# Corpus IroSvA original, usado SOLO para auditar integridad/IDs del train.
RAW_DATA_DIR = Path("../data")

RESULTS_DIR = Path("../data/resultados_beto")
PROGRESS_DIR = RESULTS_DIR / "seguimiento"
SPLIT_DIR = RESULTS_DIR / "splits_grouped_5x3_v3"

TEXT_COLUMNS = {
    "BASE": "MESSAGE_TRANSFORMER_BASE",
    "SOCIAL": "MESSAGE_TRANSFORMER_SOCIAL",
}
LABEL_COL = "IS_IRONIC"
ID_COL = "ID"
ORIGINAL_COL = "MESSAGE"
GROUP_COL = "GROUP_ID"

EXPECTED_N = {"mx": 2397, "es": 2396, "cu": 2400}
MAX_LENGTH = 128
RANDOM_STATE = 42
CLASSIFIER_DROPOUT = 0.1

# ------------------------------------------------------------
# "prueba" sirve para validar el pipeline rápidamente.
# "completo" = experimento metodológico definitivo.
# ------------------------------------------------------------
MODO = "prueba"

if MODO == "prueba":
    VARIANTES = ["mx"]
    N_OUTER = 2
    N_INNER = 2
    N_TRIALS = 2
    N_STARTUP_TRIALS = 1
elif MODO == "completo":
    VARIANTES = ["mx", "es", "cu"]
    N_OUTER = 5
    N_INNER = 3
    N_TRIALS = 20
    N_STARTUP_TRIALS = 5
else:
    raise ValueError("MODO debe ser 'prueba' o 'completo'.")

RUN_TAG = "beto_grouped_5x3_tpe20_final_uniforme"
RUN_DIR = PROGRESS_DIR / RUN_TAG / MODO
TMP_DIR = RUN_DIR / "tmp"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
FIT_STATE_DIR = RUN_DIR / "fit_states"
CURVE_DIR = RUN_DIR / "curvas"
GRAPH_DIR = RUN_DIR / "graficas_mejores_trials"
AUDIT_DIR = RUN_DIR / "auditoria"

for carpeta in [RESULTS_DIR, PROGRESS_DIR, SPLIT_DIR, RUN_DIR, TMP_DIR,
                CHECKPOINT_DIR, FIT_STATE_DIR, CURVE_DIR, GRAPH_DIR, AUDIT_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)

# Flags de seguridad.
EJECUTAR_NESTED = False
EJECUTAR_OPTIMIZACION_FINAL = False
EJECUTAR_TEST_OFICIAL = False

# Si BASE o SOCIAL generan una entrada idéntica en train y test oficial,
# detener antes de entrenar para revisar el caso. No se leen etiquetas de test.
FAIL_ON_BETO_TRAIN_TEST_EQUIVALENCE = True

# Los checkpoints de un fit COMPLETADO se borran para no llenar el disco.
# Si se interrumpe, el checkpoint del fit EN CURSO permanece y se reanuda.
CLEAN_COMPLETED_CHECKPOINTS = True

HEARTBEAT_SEGUNDOS = 30
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16
if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")

fits_teoricos = len(VARIANTES) * N_OUTER * (N_TRIALS * N_INNER + 1)
print(f"[CONFIG][BETO] modo={MODO} | variantes={VARIANTES}")
print(f"[CONFIG][BETO] Nested={N_OUTER}×{N_INNER} | trials={N_TRIALS} | startup={N_STARTUP_TRIALS}")
print(f"[CONFIG][BETO] fine-tunings teóricos={fits_teoricos} | nested_activo={EJECUTAR_NESTED}")
print(f"[CONFIG][BETO] RUN_DIR={RUN_DIR.resolve()}")

## 3. Reproducibilidad

In [ ]:
def fijar_semilla(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)


def liberar_memoria():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


fijar_semilla(RANDOM_STATE)

## 4. Carga del train Transformer

In [ ]:
def cargar_train_variante(variante):
    ruta = DATA_DIR / f"train_transformers_{variante}.csv"
    if not ruta.exists():
        raise FileNotFoundError(
            f"No existe {ruta}. Ejecuta primero el preprocessing Transformer final."
        )
    df = pd.read_csv(ruta)
    requeridas = [ID_COL, ORIGINAL_COL, LABEL_COL, *TEXT_COLUMNS.values()]
    faltantes = [c for c in requeridas if c not in df.columns]
    if faltantes:
        raise ValueError(f"{variante}: faltan columnas {faltantes}")

    df[ID_COL] = df[ID_COL].astype(str)
    df[LABEL_COL] = df[LABEL_COL].astype(int)
    for col in [ORIGINAL_COL, *TEXT_COLUMNS.values()]:
        df[col] = df[col].fillna("").astype(str)
    return df.reset_index(drop=True)


for variante in VARIANTES:
    tmp = cargar_train_variante(variante)
    print(
        f"{variante.upper()}: {len(tmp)} filas | "
        f"clases={tmp[LABEL_COL].value_counts().sort_index().to_dict()}"
    )

## 5. Auditoría de integridad y construcción independiente de `GROUP_ID`

Los grupos se construyen **antes** del Nested CV y son independientes de la opción `BASE`/`SOCIAL` que Optuna seleccione.

Dos registros quedan conectados al mismo grupo si comparten al menos una de estas representaciones que BETO puede recibir o que identifican el mismo texto original:

1. texto original exacto, normalizando únicamente espacios;
2. entrada `MESSAGE_TRANSFORMER_BASE` idéntica;
3. entrada `MESSAGE_TRANSFORMER_SOCIAL` idéntica.

La normalización histórica usada en los experimentos de ML (`KEY_ML_EQUIV`) **se calcula solo para auditoría** y **NO forma parte de `GROUP_ID`**. Así evitamos asumir que dos textos que se vuelven iguales bajo el preprocessing de ML son necesariamente duplicados reales para BETO.

Se usan **componentes conexas**: si A coincide con B por una representación de BETO y B coincide con C por otra, A/B/C permanecen en el mismo grupo.

In [ ]:
def colapsar_espacios(texto):
    return re.sub(r"\s+", " ", str(texto)).strip()


def normalizar_equivalencia_ml(texto):
    """Normalización histórica de ML usada SOLO para auditoría.

    IMPORTANTE: KEY_ML_EQUIV no participa en GROUP_ID y no se usa como entrada de BETO.
    """
    texto = str(texto)
    texto = re.sub(r"http\S+|www\S+", "[URL]", texto)
    texto = re.sub(r"@\w+", "[USER]", texto)
    texto = emoji.demojize(texto, language="es")
    texto = re.sub(r"(.)\1{2,}", r"\1\1", texto)
    texto = re.sub(r"[^\w\s\[\]áéíóúüñÁÉÍÓÚÜÑ]", " ", texto)
    texto = texto.lower()
    return colapsar_espacios(texto)


class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


def construir_grupos_equivalencia(df, variante):
    trabajo = df.copy().reset_index(drop=True)
    trabajo["KEY_ORIGINAL"] = trabajo[ORIGINAL_COL].map(colapsar_espacios)
    trabajo["KEY_ML_EQUIV"] = trabajo[ORIGINAL_COL].map(normalizar_equivalencia_ml)
    trabajo["KEY_BASE"] = trabajo[TEXT_COLUMNS["BASE"]].map(colapsar_espacios)
    trabajo["KEY_SOCIAL"] = trabajo[TEXT_COLUMNS["SOCIAL"]].map(colapsar_espacios)

    # GROUP_ID se construye únicamente con representaciones relevantes para BETO.
    # KEY_ML_EQUIV queda deliberadamente fuera.
    uf = UnionFind(len(trabajo))
    key_cols_group = ["KEY_ORIGINAL", "KEY_BASE", "KEY_SOCIAL"]

    for col in key_cols_group:
        first_seen = {}
        for idx, key in enumerate(trabajo[col].tolist()):
            if key == "":
                continue
            if key in first_seen:
                uf.union(idx, first_seen[key])
            else:
                first_seen[key] = idx

    componentes = {}
    for i in range(len(trabajo)):
        componentes.setdefault(uf.find(i), []).append(i)

    group_ids = [None] * len(trabajo)
    for miembros in componentes.values():
        ids = sorted(trabajo.loc[miembros, ID_COL].astype(str).tolist())
        digest = hashlib.sha1("|".join(ids).encode("utf-8")).hexdigest()[:12]
        gid = f"{variante}_{digest}"
        for i in miembros:
            group_ids[i] = gid

    trabajo[GROUP_COL] = group_ids
    return trabajo


def auditar_integridad_fuente(df, variante):
    """Audita correspondencia por ID con el IroSvA original y explica faltantes.

    Si el raw existe, cualquier ID procesado inexistente en raw o cualquier cambio en
    MESSAGE/IS_IRONIC para un ID presente se considera una inconsistencia fuerte.
    """
    ruta_raw = RAW_DATA_DIR / f"irosva.{variante}.training.csv"
    if not ruta_raw.exists():
        return pd.DataFrame(), f"No se encontró {ruta_raw}; no se pudo reconstruir el motivo de faltantes."

    raw = pd.read_csv(ruta_raw)
    requeridas = [ID_COL, ORIGINAL_COL, LABEL_COL]
    if any(c not in raw.columns for c in requeridas):
        return pd.DataFrame(), f"{ruta_raw} no contiene {requeridas}."

    raw[ID_COL] = raw[ID_COL].astype(str)
    raw[ORIGINAL_COL] = raw[ORIGINAL_COL].fillna("").astype(str)
    raw[LABEL_COL] = raw[LABEL_COL].astype(int)

    if raw[ID_COL].duplicated().any():
        detalle = raw[raw[ID_COL].duplicated(keep=False)].copy()
        detalle.to_csv(AUDIT_DIR / f"ids_duplicados_raw_{variante}.csv", index=False)
        raise ValueError(f"{variante}: el raw contiene IDs duplicados. Revisa la auditoría.")

    ids_raw = set(raw[ID_COL])
    ids_proc = set(df[ID_COL].astype(str))
    desconocidos = df[~df[ID_COL].astype(str).isin(ids_raw)].copy()
    if not desconocidos.empty:
        desconocidos.to_csv(AUDIT_DIR / f"ids_procesados_no_en_raw_{variante}.csv", index=False)
        raise ValueError(f"{variante}: hay IDs procesados que no existen en el raw.")

    # Verifica que MESSAGE e IS_IRONIC de los IDs conservados no hayan cambiado.
    comp = df[[ID_COL, ORIGINAL_COL, LABEL_COL]].merge(
        raw[[ID_COL, ORIGINAL_COL, LABEL_COL]],
        on=ID_COL,
        how="left",
        suffixes=("_PROC", "_RAW"),
        validate="one_to_one",
    )
    mismatch = comp[
        (comp[f"{ORIGINAL_COL}_PROC"] != comp[f"{ORIGINAL_COL}_RAW"])
        | (comp[f"{LABEL_COL}_PROC"] != comp[f"{LABEL_COL}_RAW"])
    ].copy()
    if not mismatch.empty:
        mismatch.to_csv(AUDIT_DIR / f"mismatch_procesado_vs_raw_{variante}.csv", index=False)
        raise ValueError(
            f"{variante}: MESSAGE o IS_IRONIC no coincide con el raw para {len(mismatch)} IDs."
        )

    faltantes = raw[~raw[ID_COL].isin(ids_proc)].copy()
    if faltantes.empty:
        return faltantes, "No faltan IDs respecto al raw y los registros conservados coinciden."

    stats_msg = raw.groupby(ORIGINAL_COL)[LABEL_COL].agg(["size", "nunique"])

    def motivo(row):
        st = stats_msg.loc[row[ORIGINAL_COL]]
        if st["size"] > 1 and st["nunique"] > 1:
            return "conflicto_etiqueta_texto_original"
        if st["size"] > 1 and st["nunique"] == 1:
            return "duplicado_texto_original_misma_etiqueta"
        return "otra_exclusion_revisar"

    faltantes["MOTIVO_PROBABLE"] = faltantes.apply(motivo, axis=1)
    return faltantes, f"{len(faltantes)} IDs del raw no están en train_transformers_{variante}.csv"


def _conflictos_label_por_key(trabajo, key_col):
    validos = trabajo[trabajo[key_col].ne("")]
    stats = validos.groupby(key_col)[LABEL_COL].agg(["size", "nunique"])
    claves = set(stats[(stats["size"] > 1) & (stats["nunique"] > 1)].index)
    detalle = validos[validos[key_col].isin(claves)].copy()
    return claves, detalle


def auditar_y_agrupar_train(variante):
    df = cargar_train_variante(variante)

    if df[ID_COL].duplicated().any():
        duplicados = df.loc[df[ID_COL].duplicated(keep=False), ID_COL].tolist()
        raise ValueError(f"{variante}: IDs duplicados en train: {duplicados[:10]}")
    if df[ORIGINAL_COL].map(colapsar_espacios).eq("").any():
        raise ValueError(f"{variante}: hay MESSAGE vacío.")
    if not set(df[LABEL_COL].unique()).issubset({0, 1}):
        raise ValueError(f"{variante}: etiquetas fuera de 0/1.")

    # Un MESSAGE original idéntico con etiquetas contradictorias es una inconsistencia fuerte.
    conflicto_original = (
        df.groupby(df[ORIGINAL_COL].map(colapsar_espacios))[LABEL_COL]
          .nunique()
          .gt(1)
    )
    if conflicto_original.any():
        claves = set(conflicto_original[conflicto_original].index)
        detalle = df[df[ORIGINAL_COL].map(colapsar_espacios).isin(claves)]
        detalle.to_csv(AUDIT_DIR / f"conflicto_original_{variante}.csv", index=False)
        raise ValueError(
            f"{variante}: quedan textos originales idénticos con etiquetas contradictorias. "
            "Revisa el CSV de auditoría antes de entrenar."
        )

    trabajo = construir_grupos_equivalencia(df, variante)

    # Auditoría de equivalencias por representación. ML_EQUIV se informa pero no agrupa.
    filas = []
    for nombre, col in [
        ("ORIGINAL", "KEY_ORIGINAL"),
        ("ML_EQUIV_AUDIT", "KEY_ML_EQUIV"),
        ("BETO_BASE", "KEY_BASE"),
        ("BETO_SOCIAL", "KEY_SOCIAL"),
    ]:
        sizes = trabajo[trabajo[col].ne("")].groupby(col).size()
        rep_keys = sizes[sizes > 1].index
        mask = trabajo[col].isin(rep_keys) & trabajo[col].ne("")
        filas.append({
            "variante": variante,
            "representacion": nombre,
            "filas_en_repeticiones": int(mask.sum()),
            "grupos_repetidos": int((sizes > 1).sum()),
        })

    # Si BETO recibiría exactamente el mismo input con etiquetas opuestas, se documenta
    # y el pre-flight bloqueará el entrenamiento.
    conflictos_beto = {}
    for nombre, col in [("BASE", "KEY_BASE"), ("SOCIAL", "KEY_SOCIAL")]:
        claves, detalle = _conflictos_label_por_key(trabajo, col)
        conflictos_beto[nombre] = len(claves)
        if not detalle.empty:
            detalle[[ID_COL, ORIGINAL_COL, LABEL_COL, GROUP_COL, col]].to_csv(
                AUDIT_DIR / f"conflictos_label_beto_{nombre.lower()}_{variante}.csv",
                index=False,
            )

    # Grupos mixtos se auditan. No se eliminan automáticamente.
    gstats = trabajo.groupby(GROUP_COL)[LABEL_COL].agg(["size", "nunique"])
    conflict_gids = set(gstats[(gstats["size"] > 1) & (gstats["nunique"] > 1)].index)
    conflictos = trabajo[trabajo[GROUP_COL].isin(conflict_gids)].copy()
    if not conflictos.empty:
        conflictos[
            [ID_COL, ORIGINAL_COL, LABEL_COL, GROUP_COL, "KEY_BASE", "KEY_SOCIAL"]
        ].to_csv(AUDIT_DIR / f"grupos_etiquetas_mixtas_{variante}.csv", index=False)

    faltantes, mensaje_faltantes = auditar_integridad_fuente(trabajo, variante)
    if not faltantes.empty:
        faltantes.to_csv(AUDIT_DIR / f"ids_faltantes_vs_raw_{variante}.csv", index=False)

    resumen = pd.DataFrame(filas)
    resumen["n_train"] = len(trabajo)
    resumen["n_group_id"] = trabajo[GROUP_COL].nunique()
    resumen["max_group_size"] = int(trabajo.groupby(GROUP_COL).size().max())
    resumen["grupos_con_labels_mixtos"] = len(conflict_gids)
    resumen["conflictos_label_beto_base"] = conflictos_beto["BASE"]
    resumen["conflictos_label_beto_social"] = conflictos_beto["SOCIAL"]

    # Guardar el mapeo estable ID -> GROUP_ID.
    trabajo[
        [ID_COL, LABEL_COL, ORIGINAL_COL, *TEXT_COLUMNS.values(), GROUP_COL]
    ].to_csv(SPLIT_DIR / f"beto_groups_{variante}.csv", index=False)

    return trabajo, resumen, conflictos, mensaje_faltantes


TRAIN_AUDITADO = {}
resumenes_grupos = []

for variante in VARIANTES:
    df_g, resumen_g, conflictos_g, msg_faltantes = auditar_y_agrupar_train(variante)
    TRAIN_AUDITADO[variante] = df_g
    resumenes_grupos.append(resumen_g)
    print(f"\n{variante.upper()} — {msg_faltantes}")
    print(
        f"n={len(df_g)} | GROUP_ID={df_g[GROUP_COL].nunique()} | "
        f"max grupo={df_g.groupby(GROUP_COL).size().max()} | "
        f"grupos con labels mixtos={df_g.groupby(GROUP_COL)[LABEL_COL].nunique().gt(1).sum()}"
    )

DF_AUDITORIA_GRUPOS = pd.concat(resumenes_grupos, ignore_index=True)
display(DF_AUDITORIA_GRUPOS)
DF_AUDITORIA_GRUPOS.to_csv(AUDIT_DIR / "auditoria_grupos_train.csv", index=False)


## 6. Auditoría train ↔ test oficial sin cargar etiquetas del test

Esta sección lee **solo** `ID`, `MESSAGE`, `BASE` y `SOCIAL` del archivo de test. `IS_IRONIC` no se carga.

- `ML_EQUIV` se reporta solo como advertencia histórica/metodológica; no convierte automáticamente una coincidencia en duplicado real para BETO.
- Si existe una coincidencia exacta en la entrada real `BASE` o `SOCIAL`, el pre-flight se detiene por defecto antes del entrenamiento.

In [ ]:
def cargar_test_sin_labels(variante):
    ruta = DATA_DIR / f"test_transformers_{variante}.csv"
    if not ruta.exists():
        raise FileNotFoundError(ruta)
    cols = [ID_COL, ORIGINAL_COL, *TEXT_COLUMNS.values()]
    df = pd.read_csv(ruta, usecols=cols)  # Deliberadamente NO carga IS_IRONIC.
    df[ID_COL] = df[ID_COL].astype(str)
    for c in [ORIGINAL_COL, *TEXT_COLUMNS.values()]:
        df[c] = df[c].fillna("").astype(str)
    df["KEY_ORIGINAL"] = df[ORIGINAL_COL].map(colapsar_espacios)
    df["KEY_ML_EQUIV"] = df[ORIGINAL_COL].map(normalizar_equivalencia_ml)
    df["KEY_BASE"] = df[TEXT_COLUMNS["BASE"]].map(colapsar_espacios)
    df["KEY_SOCIAL"] = df[TEXT_COLUMNS["SOCIAL"]].map(colapsar_espacios)
    return df


def detalle_solapamiento(tr, te, key_col, variante, nombre):
    comunes = (set(tr[key_col]) & set(te[key_col])) - {""}
    if not comunes:
        return pd.DataFrame(columns=["variante", "tipo", "KEY", "ID_TRAIN", "ID_TEST"])
    a = tr[tr[key_col].isin(comunes)][[ID_COL, key_col, ORIGINAL_COL]].rename(
        columns={ID_COL: "ID_TRAIN", ORIGINAL_COL: "MESSAGE_TRAIN", key_col: "KEY"}
    )
    b = te[te[key_col].isin(comunes)][[ID_COL, key_col, ORIGINAL_COL]].rename(
        columns={ID_COL: "ID_TEST", ORIGINAL_COL: "MESSAGE_TEST", key_col: "KEY"}
    )
    out = a.merge(b, on="KEY", how="inner")
    out.insert(0, "tipo", nombre)
    out.insert(0, "variante", variante)
    return out


AUDITORIA_TEST = []
DETALLES_TEST = []

for variante in VARIANTES:
    tr = TRAIN_AUDITADO[variante]
    te = cargar_test_sin_labels(variante)

    id_overlap = set(tr[ID_COL]) & set(te[ID_COL])
    fila = {"variante": variante, "train_n": len(tr), "test_n": len(te), "id_overlap": len(id_overlap)}

    for nombre, key_col in [
        ("ORIGINAL", "KEY_ORIGINAL"),
        ("ML_EQUIV", "KEY_ML_EQUIV"),
        ("BETO_BASE", "KEY_BASE"),
        ("BETO_SOCIAL", "KEY_SOCIAL"),
    ]:
        detalle = detalle_solapamiento(tr, te, key_col, variante, nombre)
        fila[f"overlap_{nombre.lower()}"] = int(detalle["KEY"].nunique()) if not detalle.empty else 0
        if not detalle.empty:
            DETALLES_TEST.append(detalle)

    AUDITORIA_TEST.append(fila)

DF_AUDITORIA_TEST = pd.DataFrame(AUDITORIA_TEST)
display(DF_AUDITORIA_TEST)
DF_AUDITORIA_TEST.to_csv(AUDIT_DIR / "auditoria_train_test_sin_labels.csv", index=False)

if DETALLES_TEST:
    DF_DETALLE_TEST = pd.concat(DETALLES_TEST, ignore_index=True)
    DF_DETALLE_TEST.to_csv(AUDIT_DIR / "detalle_solapamientos_train_test_sin_labels.csv", index=False)
else:
    DF_DETALLE_TEST = pd.DataFrame()

print("Etiquetas del test cargadas en memoria: NO")

## 7. Pre-flight metodológico obligatorio

In [ ]:
def comprobar_preflight():
    print("=" * 78)
    print("PRE-FLIGHT — BETO GROUPED NESTED CV")
    print("=" * 78)

    if MODO == "completo":
        esperado = (N_OUTER, N_INNER, N_TRIALS, N_STARTUP_TRIALS)
        if esperado != (5, 3, 20, 5):
            raise ValueError(f"Configuración inesperada: {esperado}; se esperaba 5×3, 20 trials, startup=5.")

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA no disponible. Verifica entorno/PyTorch antes de entrenar BETO.")

    for variante in VARIANTES:
        df = TRAIN_AUDITADO[variante]
        if MODO == "completo" and len(df) != EXPECTED_N[variante]:
            raise ValueError(
                f"{variante.upper()}: n={len(df)}, pero se esperaba {EXPECTED_N[variante]}. "
                f"Revisa {AUDIT_DIR / f'ids_faltantes_vs_raw_{variante}.csv'} antes de entrenar."
            )
        if df[GROUP_COL].isna().any():
            raise ValueError(f"{variante}: GROUP_ID nulo.")
        if df[ID_COL].duplicated().any():
            raise ValueError(f"{variante}: IDs duplicados.")

        resumen_v = DF_AUDITORIA_GRUPOS[DF_AUDITORIA_GRUPOS["variante"] == variante]
        n_conf_base = int(resumen_v["conflictos_label_beto_base"].max())
        n_conf_social = int(resumen_v["conflictos_label_beto_social"].max())
        if n_conf_base or n_conf_social:
            print(
                f"[AUDIT][BETO][{variante.upper()}] ADVERTENCIA: entradas BETO idénticas con etiquetas opuestas "
                f"(BASE={n_conf_base}, SOCIAL={n_conf_social}). No es leakage si el GROUP_ID permanece íntegro; "
                "se conserva y se documenta como ruido/ambigüedad de anotación."
            )

    # La expectativa documentada es que no existan IDs ni textos originales exactos entre train/test.
    if (DF_AUDITORIA_TEST["id_overlap"] > 0).any():
        raise ValueError("Se detectaron IDs compartidos entre train y test. Revisar auditoría.")
    if (DF_AUDITORIA_TEST["overlap_original"] > 0).any():
        raise ValueError("Se detectaron MESSAGE originales exactos entre train y test. Revisar auditoría.")

    # Coincidencias ML_EQUIV se informan, no se usan para detener BETO por sí solas.
    n_ml = int(DF_AUDITORIA_TEST["overlap_ml_equiv"].sum())
    if n_ml:
        print(f"ADVERTENCIA: {n_ml} claves ML_EQUIV train↔test. No se consideran automáticamente duplicados de BETO.")

    n_beto = int(
        DF_AUDITORIA_TEST["overlap_beto_base"].sum()
        + DF_AUDITORIA_TEST["overlap_beto_social"].sum()
    )
    if n_beto and FAIL_ON_BETO_TRAIN_TEST_EQUIVALENCE:
        raise ValueError(
            f"Hay {n_beto} coincidencias de entrada BETO (BASE/SOCIAL) entre train y test. "
            f"Revisa {AUDIT_DIR / 'detalle_solapamientos_train_test_sin_labels.csv'} antes de entrenar. "
            "No se utilizaron etiquetas del test."
        )

    print("GPU:", torch.cuda.get_device_name(0))
    print(f"Nested: {N_OUTER}×{N_INNER} | Optuna={N_TRIALS} | startup={N_STARTUP_TRIALS}")
    print("Test labels: NO CARGADAS")
    print("PRE-FLIGHT DE DATOS: OK")


comprobar_preflight()

## 8. Crear y guardar las particiones `StratifiedGroupKFold`

Las particiones se generan una sola vez de forma determinista. El entrenamiento posterior usa estos índices precomputados, no vuelve a sortear folds.

El manifiesto largo guarda para cada `Outer Fold`:

- qué filas son `outer_train` y `outer_test`;
- para las filas de `outer_train`, en qué `inner_val_fold` actúan como validación.

In [ ]:
def crear_splits_variante(df, variante):
    y = df[LABEL_COL].to_numpy(dtype=np.int64)
    groups = df[GROUP_COL].astype(str).to_numpy()
    indices = np.arange(len(df), dtype=int)

    outer_cv = StratifiedGroupKFold(
        n_splits=N_OUTER, shuffle=True, random_state=RANDOM_STATE
    )

    split_obj = {}
    manifest_rows = []
    diag_rows = []
    outer_test_seen = np.zeros(len(df), dtype=int)

    for outer_fold, (idx_train, idx_outer) in enumerate(
        outer_cv.split(indices, y, groups), start=1
    ):
        idx_train = indices[idx_train]
        idx_outer = indices[idx_outer]

        # Cero fuga de GROUP_ID entre Outer Train y Outer Test.
        g_train = set(groups[idx_train])
        g_outer = set(groups[idx_outer])
        if not g_train.isdisjoint(g_outer):
            raise AssertionError(f"{variante} Outer {outer_fold}: GROUP_ID compartido train/test.")
        if set(idx_train) & set(idx_outer):
            raise AssertionError(f"{variante} Outer {outer_fold}: índice compartido train/test.")
        outer_test_seen[idx_outer] += 1

        # Inner SGKF se hace SOLO sobre Outer Train y se convierte a índices globales.
        inner_cv = StratifiedGroupKFold(
            n_splits=N_INNER, shuffle=True, random_state=RANDOM_STATE + outer_fold
        )
        local_y = y[idx_train]
        local_g = groups[idx_train]
        inner_splits = []
        inner_assignment = {}

        for inner_fold, (itr_local, iva_local) in enumerate(
            inner_cv.split(np.arange(len(idx_train)), local_y, local_g), start=1
        ):
            itr = idx_train[itr_local]
            iva = idx_train[iva_local]
            if not set(groups[itr]).isdisjoint(set(groups[iva])):
                raise AssertionError(
                    f"{variante} Outer {outer_fold} Inner {inner_fold}: GROUP_ID compartido train/val."
                )
            if set(itr) & set(iva):
                raise AssertionError(
                    f"{variante} Outer {outer_fold} Inner {inner_fold}: índice compartido train/val."
                )
            for i in iva:
                if int(i) in inner_assignment:
                    raise AssertionError("Una fila fue validación en más de un Inner Fold.")
                inner_assignment[int(i)] = inner_fold
            inner_splits.append((itr, iva))

            diag_rows.append({
                "variante": variante,
                "outer_fold": outer_fold,
                "inner_fold": inner_fold,
                "split": "inner_val",
                "n": len(iva),
                "n_groups": len(set(groups[iva])),
                "pct_ironic": float(y[iva].mean()),
            })

        if set(inner_assignment) != set(map(int, idx_train)):
            raise AssertionError(
                f"{variante} Outer {outer_fold}: no todas las filas de Outer Train aparecen una vez como Inner Val."
            )

        split_obj[outer_fold] = {
            "idx_train": idx_train,
            "idx_outer": idx_outer,
            "inner": inner_splits,
        }

        diag_rows.extend([
            {
                "variante": variante, "outer_fold": outer_fold, "inner_fold": np.nan,
                "split": "outer_train", "n": len(idx_train),
                "n_groups": len(set(groups[idx_train])), "pct_ironic": float(y[idx_train].mean()),
            },
            {
                "variante": variante, "outer_fold": outer_fold, "inner_fold": np.nan,
                "split": "outer_test", "n": len(idx_outer),
                "n_groups": len(set(groups[idx_outer])), "pct_ironic": float(y[idx_outer].mean()),
            },
        ])

        outer_set = set(map(int, idx_outer))
        for i in indices:
            manifest_rows.append({
                "variante": variante,
                "row_index": int(i),
                "ID": df.loc[i, ID_COL],
                "GROUP_ID": groups[i],
                "outer_fold": outer_fold,
                "outer_role": "outer_test" if int(i) in outer_set else "outer_train",
                "inner_val_fold": np.nan if int(i) in outer_set else int(inner_assignment[int(i)]),
            })

    if not np.all(outer_test_seen == 1):
        raise AssertionError(f"{variante}: cada fila debe aparecer exactamente una vez en Outer Test.")

    manifest = pd.DataFrame(manifest_rows)
    diagnostics = pd.DataFrame(diag_rows)
    return split_obj, manifest, diagnostics


def guardar_o_verificar_csv(df, ruta, sort_cols):
    ruta = Path(ruta)
    normal = df.sort_values(sort_cols).reset_index(drop=True)
    if ruta.exists():
        previo = pd.read_csv(ruta)
        # Comparación robusta de columnas y valores básicos.
        previo = previo[normal.columns].sort_values(sort_cols).reset_index(drop=True)
        a = normal.copy()
        b = previo.copy()
        for c in a.columns:
            if a[c].dtype.kind in "fc":
                a[c] = pd.to_numeric(a[c], errors="coerce")
                b[c] = pd.to_numeric(b[c], errors="coerce")
        if not a.fillna("<NA>").astype(str).equals(b.fillna("<NA>").astype(str)):
            raise ValueError(f"Las particiones regeneradas no coinciden con {ruta}. No sobrescribir.")
    else:
        normal.to_csv(ruta, index=False)


SPLITS = {}
DIAGNOSTICOS_SPLITS = []

for variante in VARIANTES:
    splits_v, manifest_v, diag_v = crear_splits_variante(TRAIN_AUDITADO[variante], variante)
    SPLITS[variante] = splits_v
    DIAGNOSTICOS_SPLITS.append(diag_v)
    guardar_o_verificar_csv(
        manifest_v,
        SPLIT_DIR / f"split_manifest_{variante}_{N_OUTER}x{N_INNER}.csv",
        ["outer_fold", "row_index"],
    )
    print(f"{variante.upper()}: particiones verificadas; cero GROUP_ID compartidos dentro de cada split.")

DF_DIAGNOSTICO_SPLITS = pd.concat(DIAGNOSTICOS_SPLITS, ignore_index=True)
DF_DIAGNOSTICO_SPLITS.to_csv(SPLIT_DIR / f"split_diagnostics_{N_OUTER}x{N_INNER}.csv", index=False)
display(DF_DIAGNOSTICO_SPLITS.head(20))

## 9. Tokenizer y datasets tokenizados

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

class EncodedDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            list(texts), truncation=True, max_length=max_length, padding=False
        )
        self.labels = np.asarray(labels, dtype=np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = int(self.labels[idx])
        return item


def preparar_cache_datasets(df):
    y = df[LABEL_COL].to_numpy(dtype=np.int64)
    cache = {
        prep: EncodedDataset(
            texts=df[col].tolist(), labels=y, tokenizer=tokenizer, max_length=MAX_LENGTH
        )
        for prep, col in TEXT_COLUMNS.items()
    }
    return cache, y

print("Tokenizer:", tokenizer.__class__.__name__)
print("Vocab size:", tokenizer.vocab_size)

## 10. Métricas y pérdida balanceada

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    pred = np.argmax(logits, axis=-1)
    return {
        "f1_macro": f1_score(labels, pred, average="macro"),
        "accuracy": accuracy_score(labels, pred),
        "precision_macro": precision_score(labels, pred, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, pred, average="macro", zero_division=0),
    }


def calcular_pesos_clase(y_train):
    clases = np.array(sorted(np.unique(y_train)))
    pesos = compute_class_weight(class_weight="balanced", classes=clases, y=y_train)
    vector = np.ones(int(clases.max()) + 1, dtype=np.float32)
    for clase, peso in zip(clases, pesos):
        vector[int(clase)] = peso
    return torch.tensor(vector, dtype=torch.float32)


class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        if self.class_weights is None:
            loss = outputs.loss
        else:
            loss = F.cross_entropy(outputs.logits, labels, weight=self.class_weights.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss

## 11. Creación de BETO

In [ ]:
def crear_modelo(seed):
    fijar_semilla(seed)
    model_log = RUN_DIR / "carga_modelos.log"
    with model_log.open("a", encoding="utf-8") as log, redirect_stdout(log), redirect_stderr(log):
        config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=2)
        config.classifier_dropout = CLASSIFIER_DROPOUT
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)
    return model

## 12. `TrainingArguments`: curvas + checkpoints reanudables

- `logging_strategy="epoch"` permite registrar **training loss** sin imprimirlo en pantalla.
- con validación: `eval_strategy/save_strategy="epoch"`, early stopping y mejor modelo por F1-Macro;
- sin validación (Outer refit): también se guarda un checkpoint por época para poder reanudar;
- `save_total_limit=2` limita el espacio del fit activo.

Cuando la versión instalada de Transformers lo permite, también se activa `restore_callback_states_from_checkpoint=True` para restaurar el estado de callbacks al reanudar.

In [ ]:
def crear_training_args(
    output_dir,
    learning_rate,
    batch_size,
    num_train_epochs,
    weight_decay,
    seed,
    con_evaluacion=True,
):
    params = {
        "output_dir": str(output_dir),
        "learning_rate": learning_rate,
        "per_device_train_batch_size": batch_size,
        "per_device_eval_batch_size": batch_size,
        "num_train_epochs": num_train_epochs,
        "weight_decay": weight_decay,
        "warmup_ratio": 0.10,
        "logging_strategy": "epoch",
        "logging_first_step": False,
        "save_strategy": "epoch",
        "save_total_limit": 2,
        "report_to": "none",
        "seed": seed,
        "data_seed": seed,
        "bf16": USE_BF16,
        "fp16": USE_FP16,
        "dataloader_num_workers": 0,
        "optim": "adamw_torch",
        "disable_tqdm": True,
    }

    if con_evaluacion:
        params.update({
            "load_best_model_at_end": True,
            "metric_for_best_model": "f1_macro",
            "greater_is_better": True,
        })
        eval_value = "epoch"
    else:
        params.update({"load_best_model_at_end": False})
        eval_value = "no"

    firma = inspect.signature(TrainingArguments.__init__)
    # En versiones recientes de Transformers esto restaura también el estado de
    # callbacks como EarlyStopping al reanudar desde checkpoint.
    if "restore_callback_states_from_checkpoint" in firma.parameters:
        params["restore_callback_states_from_checkpoint"] = True

    if "eval_strategy" in firma.parameters:
        params["eval_strategy"] = eval_value
    elif "evaluation_strategy" in firma.parameters:
        params["evaluation_strategy"] = eval_value

    params = {k: v for k, v in params.items() if k in firma.parameters}
    return TrainingArguments(**params)

## 13. Espacio de búsqueda Optuna

In [ ]:
def sugerir_configuracion(trial):
    return {
        "preprocessing": trial.suggest_categorical("preprocessing", ["BASE", "SOCIAL"]),
        "learning_rate": trial.suggest_categorical("learning_rate", [2e-5, 3e-5, 5e-5]),
        "batch_size": trial.suggest_categorical("batch_size", [16, 32]),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 3, 5),
        "weight_decay": trial.suggest_categorical("weight_decay", [0.0, 0.01, 0.05]),
        "loss_mode": trial.suggest_categorical("loss_mode", ["standard", "balanced"]),
    }

## 14. Monitor, guardado atómico y firma de experimento

In [ ]:
class Monitor:
    def __init__(self, carpeta, total_estimado, ya_completados=0):
        self.carpeta = Path(carpeta)
        self.log_path = self.carpeta / "monitor.log"
        self.inicio = time.monotonic()
        self.total = int(total_estimado)
        self.hechos = int(ya_completados)
        self.hechos_inicio = int(ya_completados)
        self.etapa = "Preparando experimento"
        self.ultima_senal = 0.0
        self.handle = display(HTML("<b>Preparando BETO...</b>"), display_id=True)
        self.registrar(f"Inicio: fits recuperados={self.hechos}/{self.total}")
        self.mostrar("Inicializando", forzar=True)

    def registrar(self, texto):
        fecha = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        with self.log_path.open("a", encoding="utf-8") as f:
            f.write(f"[{fecha}] {texto}\n")

    def mostrar(self, etapa=None, forzar=False):
        if etapa is not None:
            self.etapa = str(etapa)
        ahora = time.monotonic()
        if not forzar and ahora - self.ultima_senal < HEARTBEAT_SEGUNDOS:
            return
        self.ultima_senal = ahora
        transcurrido = max(0.0, ahora - self.inicio)
        realizados = max(0, self.hechos - self.hechos_inicio)
        faltan = max(0, self.total - self.hechos)
        eta = "calculando..." if realizados < 2 else f"~{(transcurrido / realizados * faltan) / 60:.0f} min"
        progreso = 100 * self.hechos / max(1, self.total)
        contenido = (
            '<div style="font-family:system-ui;padding:12px;border:1px solid #777;border-radius:8px;max-width:800px">'
            '<b>BETO — seguimiento en vivo</b><br>'
            f'Etapa: <b>{html.escape(self.etapa)}</b><br>'
            f'Fine-tunings completados: {self.hechos}/{self.total} ({progreso:.1f}%)<br>'
            f'<progress value="{self.hechos}" max="{max(self.total,1)}" style="width:100%"></progress><br>'
            f'Tiempo: {transcurrido/60:.1f} min | ETA orientativo: {eta}<br>'
            f'Última señal: {datetime.now().strftime("%H:%M:%S")}<br>'
            f'Log: {html.escape(str(self.log_path))}</div>'
        )
        self.handle.update(HTML(contenido))

    def iniciar(self, etapa):
        self.registrar(etapa)
        self.mostrar(etapa, forzar=True)

    def pulso(self):
        if time.monotonic() - self.ultima_senal >= HEARTBEAT_SEGUNDOS:
            self.registrar(f"ACTIVO: {self.etapa}")
        self.mostrar(forzar=False)

    def fit_terminado(self, texto="Fit finalizado"):
        self.hechos += 1
        self.registrar(texto)
        self.mostrar(texto, forzar=True)


class PulsoTrainer(TrainerCallback):
    def __init__(self, monitor):
        self.monitor = monitor
    def on_step_end(self, args, state, control, **kwargs):
        if self.monitor is not None:
            self.monitor.pulso()
        return control


def entrenador_silencioso(trainer):
    trainer.remove_callback(PrinterCallback)
    trainer.remove_callback(ProgressCallback)
    return trainer


def json_atomico(obj, ruta):
    ruta = Path(ruta)
    tmp = ruta.with_name(ruta.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, ruta)


def csv_atomico(df, ruta):
    ruta = Path(ruta)
    tmp = ruta.with_name(ruta.name + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, ruta)


def firma_experimento():
    datasets = {}
    manifests = {}
    for v in VARIANTES:
        rp = DATA_DIR / f"train_transformers_{v}.csv"
        sp = SPLIT_DIR / f"split_manifest_{v}_{N_OUTER}x{N_INNER}.csv"
        datasets[v] = hashlib.sha256(rp.read_bytes()).hexdigest()
        manifests[v] = hashlib.sha256(sp.read_bytes()).hexdigest()
    return {
        "model_name": MODEL_NAME,
        "modo": MODO,
        "variantes": VARIANTES,
        "outer": N_OUTER,
        "inner": N_INNER,
        "trials": N_TRIALS,
        "n_startup_trials": N_STARTUP_TRIALS,
        "run_tag": RUN_TAG,
        "max_length": MAX_LENGTH,
        "seed": RANDOM_STATE,
        "dropout": CLASSIFIER_DROPOUT,
        "text_columns": TEXT_COLUMNS,
        "group_algorithm": "connected_components_original_base_social_v3_ml_equiv_audit_only",
        "datasets_sha256": datasets,
        "split_manifests_sha256": manifests,
    }


def comprobar_firma_experimento():
    ruta = RUN_DIR / "config_ejecucion.json"
    actual = firma_experimento()
    if ruta.exists():
        anterior = json.loads(ruta.read_text(encoding="utf-8"))
        if anterior != actual:
            raise ValueError(
                f"{RUN_DIR} contiene una corrida con configuración/datos distintos. Cambia RUN_TAG; no mezcles resultados."
            )
    else:
        json_atomico(actual, ruta)


def contar_fit_states_completos():
    return len(list(FIT_STATE_DIR.rglob("*_complete.json")))

## 15. Curvas y estado reanudable de cada Inner Fit

In [ ]:
def config_hash(cfg):
    payload = json.dumps(cfg, sort_keys=True, separators=(",", ":"))
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()[:14]


def extraer_curva(log_history):
    filas = {}
    for x in log_history:
        if "epoch" not in x:
            continue
        ep = round(float(x["epoch"]), 6)
        fila = filas.setdefault(ep, {"epoch": ep})
        if "loss" in x:
            fila["train_loss"] = float(x["loss"])
        if "eval_loss" in x:
            fila["val_loss"] = float(x["eval_loss"])
        if "eval_f1_macro" in x:
            fila["val_f1_macro"] = float(x["eval_f1_macro"])
        if "eval_accuracy" in x:
            fila["eval_accuracy"] = float(x["eval_accuracy"])
    if not filas:
        return pd.DataFrame(columns=["epoch", "train_loss", "val_loss", "val_f1_macro"])
    return pd.DataFrame(list(filas.values())).sort_values("epoch").reset_index(drop=True)


def obtener_mejor_epoca_desde_curva(curva, max_epochs):
    if curva.empty or "val_f1_macro" not in curva or curva["val_f1_macro"].dropna().empty:
        return int(max_epochs)
    row = curva.loc[curva["val_f1_macro"].idxmax()]
    return max(1, min(int(max_epochs), int(round(float(row["epoch"])))))


def rutas_inner_fit(variante, etiqueta, cfg, inner_fold):
    h = config_hash(cfg)
    base = f"{variante}_{etiqueta}_{h}_inner{inner_fold}"
    return {
        "hash": h,
        "state": FIT_STATE_DIR / f"{base}_complete.json",
        "curve": CURVE_DIR / f"{base}.csv",
        "ckpt": CHECKPOINT_DIR / variante / etiqueta / h / f"inner{inner_fold}",
    }


def cargar_estado_inner_si_completo(rutas):
    if not rutas["state"].exists():
        return None
    return json.loads(rutas["state"].read_text(encoding="utf-8"))


def ejecutar_inner_fit_reanudable(
    cache, y, idx_train, idx_val, cfg, variante, etiqueta, inner_fold, seed_modelo, monitor=None
):
    rutas = rutas_inner_fit(variante, etiqueta, cfg, inner_fold)
    previo = cargar_estado_inner_si_completo(rutas)
    if previo is not None:
        if monitor:
            monitor.registrar(
                f"REUTILIZADO {variante.upper()} {etiqueta} cfg={rutas['hash']} inner={inner_fold}: "
                f"F1={previo['score']:.4f}"
            )
        return float(previo["score"]), int(previo["best_epoch"]), rutas

    rutas["ckpt"].mkdir(parents=True, exist_ok=True)
    dataset_completo = cache[cfg["preprocessing"]]
    train_ds = Subset(dataset_completo, list(map(int, idx_train)))
    val_ds = Subset(dataset_completo, list(map(int, idx_val)))

    if monitor:
        monitor.iniciar(
            f"{variante.upper()} {etiqueta} | cfg={rutas['hash']} | Inner {inner_fold}/{N_INNER}"
        )

    model = crear_modelo(seed_modelo)
    class_weights = calcular_pesos_clase(y[idx_train]) if cfg["loss_mode"] == "balanced" else None
    args = crear_training_args(
        output_dir=rutas["ckpt"],
        learning_rate=cfg["learning_rate"],
        batch_size=cfg["batch_size"],
        num_train_epochs=cfg["num_train_epochs"],
        weight_decay=cfg["weight_decay"],
        seed=seed_modelo,
        con_evaluacion=True,
    )
    callbacks = [EarlyStoppingCallback(early_stopping_patience=1)]
    if monitor:
        callbacks.append(PulsoTrainer(monitor))

    trainer = entrenador_silencioso(WeightedTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=callbacks,
        class_weights=class_weights,
    ))

    ultimo = get_last_checkpoint(str(rutas["ckpt"]))
    if ultimo and monitor:
        monitor.registrar(f"Reanudando fit desde {ultimo}")
    trainer.train(resume_from_checkpoint=ultimo if ultimo else None)

    # Congelar primero la historia REAL del entrenamiento. Con load_best_model_at_end=True,
    # una llamada posterior a evaluate() usa el mejor checkpoint recargado y podría añadir
    # al log una métrica que no corresponde a la trayectoria cronológica de esa época.
    curva = extraer_curva(trainer.state.log_history)
    csv_atomico(curva, rutas["curve"])
    best_epoch = obtener_mejor_epoca_desde_curva(curva, cfg["num_train_epochs"])

    # Trainer ya conserva la métrica asociada al mejor checkpoint. Solo se evalúa de nuevo
    # como fallback si una versión concreta de Transformers no la dejó disponible.
    if trainer.state.best_metric is not None:
        score = float(trainer.state.best_metric)
    else:
        metrics = trainer.evaluate()
        score = float(metrics["eval_f1_macro"])

    json_atomico({
        "status": "complete",
        "score": score,
        "best_epoch": int(best_epoch),
        "config": cfg,
        "config_hash": rutas["hash"],
        "inner_fold": int(inner_fold),
        "seed": int(seed_modelo),
        "checkpoint_used_to_resume": ultimo,
    }, rutas["state"])

    del trainer, model
    liberar_memoria()
    if CLEAN_COMPLETED_CHECKPOINTS:
        shutil.rmtree(rutas["ckpt"], ignore_errors=True)

    if monitor:
        monitor.fit_terminado(
            f"{variante.upper()} {etiqueta} cfg={rutas['hash']} Inner {inner_fold}: F1={score:.4f}"
        )
    return score, best_epoch, rutas

## 16. Optuna Inner CV 3-fold con reanudación

In [ ]:
def ruta_estudio(variante, etiqueta):
    return RUN_DIR / f"beto_{variante}_{etiqueta}.db"


def reencolar_trials_interrumpidos(study, monitor=None):
    """Si el proceso murió dentro de un trial, marca el trial RUNNING como FAIL y
    reencola sus mismos parámetros. Los Inner Fits completos/checkpoint activo se reutilizan.
    """
    running = [t for t in study.get_trials(deepcopy=False) if t.state == optuna.trial.TrialState.RUNNING]
    for t in running:
        params = dict(t.params)
        if params:
            study.enqueue_trial(params, user_attrs={"resume_of_trial": int(t.number)})
        try:
            study.tell(t.number, state=optuna.trial.TrialState.FAIL)
        except Exception as e:
            if monitor:
                monitor.registrar(f"No se pudo marcar trial RUNNING {t.number} como FAIL: {e}")
        if monitor:
            monitor.registrar(f"Trial interrumpido {t.number}: parámetros reencolados para reanudación.")


def optimizar_con_splits(
    cache, y, inner_splits, variante, etiqueta_busqueda, n_trials, seed_cv, monitor=None
):
    etiqueta = str(etiqueta_busqueda)
    db = ruta_estudio(variante, etiqueta)
    sampler_path = RUN_DIR / f"sampler_{variante}_{etiqueta}.pkl"

    if sampler_path.exists():
        with sampler_path.open("rb") as f:
            sampler = pickle.load(f)
    else:
        sampler = optuna.samplers.TPESampler(
            seed=seed_cv,
            n_startup_trials=N_STARTUP_TRIALS,
        )

    study = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        study_name=f"BETO_{variante}_{etiqueta}",
        storage=f"sqlite:///{db.resolve().as_posix()}",
        load_if_exists=True,
    )
    reencolar_trials_interrumpidos(study, monitor=monitor)

    def objective(trial):
        cfg = sugerir_configuracion(trial)
        scores, epochs = [], []

        for inner_fold, (idx_train, idx_val) in enumerate(inner_splits, start=1):
            seed_modelo = RANDOM_STATE + seed_cv * 100 + inner_fold  # misma semilla por Inner Fold en todos los trials
            score, best_epoch, _ = ejecutar_inner_fit_reanudable(
                cache=cache,
                y=y,
                idx_train=np.asarray(idx_train, dtype=int),
                idx_val=np.asarray(idx_val, dtype=int),
                cfg=cfg,
                variante=variante,
                etiqueta=etiqueta,
                inner_fold=inner_fold,
                seed_modelo=seed_modelo,
                monitor=monitor,
            )
            scores.append(score)
            epochs.append(best_epoch)

        std_score = float(np.std(scores, ddof=1)) if len(scores) > 1 else 0.0
        refit_epochs = int(np.clip(round(np.median(epochs)), 1, cfg["num_train_epochs"]))
        trial.set_user_attr("inner_f1_std", std_score)
        trial.set_user_attr("best_epochs", [int(x) for x in epochs])
        trial.set_user_attr("refit_epochs", refit_epochs)
        trial.set_user_attr("config_hash", config_hash(cfg))
        return float(np.mean(scores))

    def callback(estudio, trial):
        tmp = sampler_path.with_suffix(".pkl.tmp")
        with tmp.open("wb") as f:
            pickle.dump(estudio.sampler, f)
        os.replace(tmp, sampler_path)
        estudio.trials_dataframe().to_csv(
            RUN_DIR / f"trials_{variante}_{etiqueta}.csv", index=False
        )
        completos = sum(t.state == optuna.trial.TrialState.COMPLETE for t in estudio.trials)
        msg = (
            f"{variante.upper()} {etiqueta} | completos={completos}/{n_trials} | "
            + (f"F1={trial.value:.4f} | mejor={estudio.best_value:.4f}" if trial.value is not None else "trial sin valor")
        )
        if monitor:
            monitor.registrar(msg)
            monitor.mostrar(msg, forzar=True)

    max_attempts = n_trials * 3
    attempts = 0
    while sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials) < n_trials:
        if attempts >= max_attempts:
            raise RuntimeError("Demasiados trials fallidos/interrumpidos. Revisa monitor.log.")
        study.optimize(
            objective,
            n_trials=1,
            callbacks=[callback],
            show_progress_bar=False,
            gc_after_trial=True,
            catch=(),
        )
        attempts += 1

    return study

## 17. Gráficas del mejor trial de cada Outer Fold

In [ ]:
def graficar_mejor_trial(variante, etiqueta, best_params):
    h = config_hash(best_params)
    curvas = []
    for inner_fold in range(1, N_INNER + 1):
        ruta = CURVE_DIR / f"{variante}_{etiqueta}_{h}_inner{inner_fold}.csv"
        if ruta.exists():
            df = pd.read_csv(ruta)
            df["inner_fold"] = inner_fold
            curvas.append(df)
    if not curvas:
        return None

    allc = pd.concat(curvas, ignore_index=True)
    agg = allc.groupby("epoch", as_index=False).agg(
        train_loss_mean=("train_loss", "mean"),
        train_loss_std=("train_loss", "std"),
        val_loss_mean=("val_loss", "mean"),
        val_loss_std=("val_loss", "std"),
        val_f1_macro_mean=("val_f1_macro", "mean"),
        val_f1_macro_std=("val_f1_macro", "std"),
        n_folds=("inner_fold", "nunique"),
    )
    agg_path = GRAPH_DIR / f"{variante}_{etiqueta}_best_{h}_curva_media.csv"
    agg.to_csv(agg_path, index=False)

    fig = plt.figure(figsize=(8, 5))
    plt.plot(agg["epoch"], agg["train_loss_mean"], marker="o", label="Training loss (media)")
    plt.plot(agg["epoch"], agg["val_loss_mean"], marker="o", label="Validation loss (media)")
    plt.xlabel("Época")
    plt.ylabel("Loss")
    plt.title(f"BETO {variante.upper()} {etiqueta} — mejor trial")
    plt.legend()
    plt.grid(alpha=0.25)
    fig.text(0.13, 0.01, "n_folds por época disponible en CSV; validación INNER (no Outer Test).", fontsize=8)
    plt.tight_layout(rect=(0, 0.04, 1, 1))
    loss_path = GRAPH_DIR / f"{variante}_{etiqueta}_best_{h}_loss.png"
    fig.savefig(loss_path, dpi=180, bbox_inches="tight")
    plt.close(fig)

    fig = plt.figure(figsize=(8, 5))
    plt.plot(agg["epoch"], agg["val_f1_macro_mean"], marker="o")
    plt.xlabel("Época")
    plt.ylabel("F1-Macro validación")
    plt.title(f"BETO {variante.upper()} {etiqueta} — F1-Macro del mejor trial")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    f1_path = GRAPH_DIR / f"{variante}_{etiqueta}_best_{h}_f1.png"
    fig.savefig(f1_path, dpi=180, bbox_inches="tight")
    plt.close(fig)

    return {"agg": str(agg_path), "loss_png": str(loss_path), "f1_png": str(f1_path)}

## 18. Outer refit reanudable

In [ ]:
def predicciones_trainer(trainer, dataset):
    salida = trainer.predict(dataset)
    logits = salida.predictions[0] if isinstance(salida.predictions, tuple) else salida.predictions
    return np.argmax(logits, axis=-1)


def refit_outer_y_evaluar_reanudable(
    cache, y, idx_train, idx_outer, best_params, refit_epochs, variante, outer_fold, seed, monitor=None
):
    prep = best_params["preprocessing"]
    state_path = FIT_STATE_DIR / f"{variante}_outer{outer_fold}_refit_complete.json"
    curve_path = CURVE_DIR / f"{variante}_outer{outer_fold}_refit.csv"
    ckpt_dir = CHECKPOINT_DIR / variante / f"outer{outer_fold}" / "refit"

    if state_path.exists():
        previo = json.loads(state_path.read_text(encoding="utf-8"))
        if monitor:
            monitor.registrar(f"Refit Outer {variante} {outer_fold} ya completo; se reutiliza.")
        return {
            "f1_train": float(previo["f1_train"]),
            "f1_outer_clase_0": float(previo["f1_outer_clase_0"]),
            "f1_outer_clase_1": float(previo["f1_outer_clase_1"]),
            "f1_outer": float(previo["f1_outer"]),
            "gap": float(previo["gap"]),
        }

    dataset_completo = cache[prep]
    train_dataset = Subset(dataset_completo, list(map(int, idx_train)))
    outer_dataset = Subset(dataset_completo, list(map(int, idx_outer)))
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    if monitor:
        monitor.iniciar(
            f"{variante.upper()} Outer {outer_fold}: refit {prep}, {refit_epochs} épocas"
        )

    model = crear_modelo(seed)
    class_weights = calcular_pesos_clase(y[idx_train]) if best_params["loss_mode"] == "balanced" else None
    args = crear_training_args(
        output_dir=ckpt_dir,
        learning_rate=best_params["learning_rate"],
        batch_size=best_params["batch_size"],
        num_train_epochs=refit_epochs,
        weight_decay=best_params["weight_decay"],
        seed=seed,
        con_evaluacion=False,
    )
    callbacks = [PulsoTrainer(monitor)] if monitor else []
    trainer = entrenador_silencioso(WeightedTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        data_collator=data_collator,
        callbacks=callbacks,
        class_weights=class_weights,
    ))
    ultimo = get_last_checkpoint(str(ckpt_dir))
    if ultimo and monitor:
        monitor.registrar(f"Reanudando Outer refit desde {ultimo}")
    trainer.train(resume_from_checkpoint=ultimo if ultimo else None)

    curva = extraer_curva(trainer.state.log_history)
    csv_atomico(curva, curve_path)

    pred_train = predicciones_trainer(trainer, train_dataset)
    pred_outer = predicciones_trainer(trainer, outer_dataset)
    f1_train = float(f1_score(y[idx_train], pred_train, average="macro"))
    f1_outer_clases = f1_score(
        y[idx_outer], pred_outer, labels=[0, 1], average=None, zero_division=0
    )
    f1_outer = float(f1_score(y[idx_outer], pred_outer, average="macro"))
    resultado = {
        "f1_train": f1_train,
        "f1_outer_clase_0": float(f1_outer_clases[0]),
        "f1_outer_clase_1": float(f1_outer_clases[1]),
        "f1_outer": f1_outer,
        "gap": f1_train - f1_outer,
    }
    json_atomico({"status": "complete", **resultado}, state_path)

    del trainer, model
    liberar_memoria()
    if CLEAN_COMPLETED_CHECKPOINTS:
        shutil.rmtree(ckpt_dir, ignore_errors=True)
    if monitor:
        monitor.fit_terminado(
            f"{variante.upper()} Outer {outer_fold} refit: F1_train={f1_train:.4f}, F1_outer={f1_outer:.4f}"
        )
    return resultado

## 19. Nested CV 5×3

In [ ]:
def ejecutar_nested_variante(variante, monitor=None):
    df = TRAIN_AUDITADO[variante]
    cache, y = preparar_cache_datasets(df)
    ruta_parcial = RUN_DIR / f"beto_nestedcv_detalle_{variante}.csv"

    if ruta_parcial.exists():
        resultados = pd.read_csv(ruta_parcial).to_dict("records")
        completados = {int(x["outer_fold"]) for x in resultados}
    else:
        resultados, completados = [], set()

    studies = {}
    for outer_fold in range(1, N_OUTER + 1):
        if outer_fold in completados:
            print(f"[{variante.upper()}][BETO][Outer {outer_fold}/{N_OUTER}] resultado recuperado; se omite.")
            continue

        split = SPLITS[variante][outer_fold]
        idx_train = split["idx_train"]
        idx_outer = split["idx_outer"]
        inner_splits = split["inner"]
        etiqueta = f"outer{outer_fold}"

        if monitor:
            monitor.iniciar(f"{variante.upper()} Outer {outer_fold}/{N_OUTER}: Optuna")

        study = optimizar_con_splits(
            cache=cache,
            y=y,
            inner_splits=inner_splits,
            variante=variante,
            etiqueta_busqueda=etiqueta,
            n_trials=N_TRIALS,
            seed_cv=RANDOM_STATE + outer_fold,
            monitor=monitor,
        )
        studies[outer_fold] = study
        best_params = study.best_params.copy()
        refit_epochs = int(study.best_trial.user_attrs["refit_epochs"])

        rutas_graf = graficar_mejor_trial(variante, etiqueta, best_params)

        resultado = refit_outer_y_evaluar_reanudable(
            cache=cache,
            y=y,
            idx_train=idx_train,
            idx_outer=idx_outer,
            best_params=best_params,
            refit_epochs=refit_epochs,
            variante=variante,
            outer_fold=outer_fold,
            seed=RANDOM_STATE + 1000 + outer_fold,
            monitor=monitor,
        )

        fila = {
            "variante": variante,
            "outer_fold": outer_fold,
            "best_preprocessing": best_params["preprocessing"],
            "f1_inner": float(study.best_value),
            "std_inner": float(study.best_trial.user_attrs["inner_f1_std"]),
            "refit_epochs": refit_epochs,
            **resultado,
            "best_params": json.dumps(best_params, ensure_ascii=False),
            "config_hash": config_hash(best_params),
            "graficas": json.dumps(rutas_graf, ensure_ascii=False) if rutas_graf else None,
        }
        resultados.append(fila)
        csv_atomico(
            pd.DataFrame(resultados).sort_values("outer_fold"), ruta_parcial
        )

        msg = (
            f"[{variante.upper()}][BETO][Outer {outer_fold}/{N_OUTER}] FINALIZADO | "
            f"prep={fila['best_preprocessing']} | "
            f"F1_inner={fila['f1_inner']:.4f}±{fila['std_inner']:.4f} | "
            f"F1_train={fila['f1_train']:.4f} | "
            f"F1_outer={fila['f1_outer']:.4f} | gap={fila['gap']:.4f}"
        )
        print(msg, flush=True)
        if monitor:
            monitor.registrar(msg)
            monitor.mostrar(msg, forzar=True)

    df_resultados = pd.DataFrame(resultados).sort_values("outer_fold").reset_index(drop=True)
    resumen = {
        "variante": variante,
        "n": len(y),
        "n_groups": int(df[GROUP_COL].nunique()),
        "f1_train_mean": float(df_resultados["f1_train"].mean()),
        "f1_nested_mean": float(df_resultados["f1_outer"].mean()),
        "f1_nested_std": float(df_resultados["f1_outer"].std(ddof=1)) if len(df_resultados) > 1 else 0.0,
        "f1_nested_min": float(df_resultados["f1_outer"].min()),
        "f1_nested_max": float(df_resultados["f1_outer"].max()),
        "gap_mean": float(df_resultados["gap"].mean()),
    }
    return df_resultados, resumen, studies

## 20. Ejecutar Nested CV

Para comenzar la corrida completa:

```python
MODO = "prueba"
EJECUTAR_NESTED = True
```

Reinicia el kernel y ejecuta desde el inicio. Si interrumpes el proceso, **no borres `RUN_DIR`**. Al ejecutar de nuevo con el mismo `RUN_TAG`, se reutilizan:

- Outer Folds ya finalizados;
- trials completos de Optuna (SQLite);
- Inner Folds completos de un trial interrumpido;
- checkpoint del Inner Fit o Outer Refit que estaba en curso.

In [ ]:
nested_detalle = {}
nested_resumen = []
nested_studies = {}

if EJECUTAR_NESTED:
    comprobar_preflight()
    comprobar_firma_experimento()

    monitor = Monitor(
        carpeta=RUN_DIR,
        total_estimado=fits_teoricos,
        ya_completados=contar_fit_states_completos(),
    )

    try:
        for variante in VARIANTES:
            detalle, resumen, studies = ejecutar_nested_variante(variante, monitor=monitor)
            nested_detalle[variante] = detalle
            nested_resumen.append(resumen)
            nested_studies[variante] = studies
            csv_atomico(
                pd.DataFrame(nested_resumen), RUN_DIR / "beto_nestedcv_resumen_parcial.csv"
            )
    except KeyboardInterrupt:
        monitor.registrar(
            "INTERRUMPIDO. Outer completos, trials, estados Inner y checkpoint del fit activo quedan guardados."
        )
        monitor.mostrar("Interrumpido: listo para reanudar", forzar=True)
        raise
    except Exception as e:
        monitor.registrar(f"ERROR: {type(e).__name__}: {e}")
        monitor.mostrar(f"Error: {type(e).__name__}; revisar monitor.log", forzar=True)
        raise
    else:
        monitor.registrar("Nested CV completo para todas las variantes.")
        monitor.mostrar("Nested CV completado", forzar=True)

    df_nested_resumen = pd.DataFrame(nested_resumen)
    csv_atomico(df_nested_resumen, RUN_DIR / "beto_nestedcv_resumen.csv")
    print("\n[BETO] RESUMEN NESTED CV")
    display(df_nested_resumen)
    for variante, detalle in nested_detalle.items():
        print(f"\n[{variante.upper()}][BETO] RESULTADOS POR OUTER FOLD")
        display(detalle.drop(columns=["best_params"], errors="ignore"))
else:
    print("Nested CV bloqueado. Auditorías y splits pueden revisarse sin entrenar.")
    print("Cuando todo esté OK, cambia EJECUTAR_NESTED=True y ejecuta desde el inicio.")

## 21. Consolidar resultados y localizar curvas

In [ ]:
if EJECUTAR_NESTED and (RUN_DIR / "beto_nestedcv_resumen.csv").exists():
    resumen = pd.read_csv(RUN_DIR / "beto_nestedcv_resumen.csv")
    display(resumen)
    print("\nCurvas de todos los Inner Fits:", CURVE_DIR.resolve())
    print("Gráficas de los mejores trials:", GRAPH_DIR.resolve())
    print("Checkpoints activos/no finalizados:", CHECKPOINT_DIR.resolve())
    print("Optuna SQLite y trials CSV:", RUN_DIR.resolve())
else:
    print("Aún no hay Nested CV completo.")

## 22. Optimización final — bloqueada por ahora

La optimización final se hará **después** de revisar los cinco Outer Folds de MX/ES/CU. Debe usar todos los registros de train, `StratifiedGroupKFold` interno, los mismos `GROUP_ID` y el mismo presupuesto de búsqueda que se decida congelar.

No se ejecuta en esta etapa para evitar mezclar evaluación Nested CV con decisiones posteriores.

In [ ]:
if EJECUTAR_OPTIMIZACION_FINAL:
    raise NotImplementedError(
        "Optimización final bloqueada en esta versión hasta revisar Nested CV. "
        "No actives este flag todavía."
    )
else:
    print("Optimización final: BLOQUEADA correctamente.")

## 23. Test oficial — bloqueado

Durante Nested CV el test oficial **no se evalúa** y sus etiquetas no se cargan. La única interacción previa es la auditoría de contaminación basada exclusivamente en textos/IDs.

In [ ]:
if EJECUTAR_TEST_OFICIAL:
    raise NotImplementedError(
        "TEST OFICIAL BLOQUEADO. Primero terminar/revisar Nested CV y congelar la configuración final."
    )
else:
    print("TEST OFICIAL: BLOQUEADO correctamente.")

## 24. Checklist antes de una corrida larga

Debe cumplirse todo lo siguiente:

- `PRE-FLIGHT DE DATOS: OK`;
- MX = 2397, ES = 2396, CU = 2400 en modo completo;
- los IDs procesados conservados existen en el raw y mantienen su `MESSAGE` e `IS_IRONIC` cuando el raw está disponible;
- no hay IDs ni `MESSAGE` originales exactos compartidos train↔test;
- `KEY_ML_EQUIV` se usa solo como auditoría y **no** para construir `GROUP_ID`;
- no existen entradas `BETO_BASE` o `BETO_SOCIAL` idénticas con etiquetas opuestas;
- no hay `GROUP_ID` compartidos entre train/validation dentro de ningún Outer/Inner Fold;
- los manifiestos están guardados en `splits_grouped_5x3_v3`;
- `EJECUTAR_OPTIMIZACION_FINAL=False`;
- `EJECUTAR_TEST_OFICIAL=False`;
- para iniciar únicamente se cambia `EJECUTAR_NESTED=True`.

Con 3 variantes, 5 Outer, 20 trials, 3 Inner y un refit por Outer, el máximo teórico es **915 fine-tunings**:

`3 × 5 × (20 × 3 + 1) = 915`.

Si no termina en una sesión, el diseño conserva SQLite de Optuna, estados de Inner Fits, Outer Folds terminados y el checkpoint del fit activo para continuar otro día.